# Demo 1 — Token economics and cost forecasting

**AI Cost Management and Token Utilization** · Module 1 · ~8 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. Token count is a **writing and serialization decision**, not a model decision.
2. The spread between the cheapest and most expensive tier is **~107×** for identical behaviour.
3. The forecast that matters is **per completed task**, and it is dominated by three multipliers:
   agent steps, long-context tier, and retry rate.

No API key is required for the tokenizer and forecasting sections — only for the optional
live round-trip at the end.

In [ ]:
%pip install -q tiktoken pandas matplotlib 2>/dev/null
import tiktoken, pandas as pd
enc = tiktoken.get_encoding('o200k_base')   # the modern BPE encoding
def ntok(s): return len(enc.encode(s))
print('tokenizer ready')

---
## 1. Tokenizer roulette — you pay for the shape, not the meaning

Same intent, five different forms. Watch the token count move while the meaning does not.

In [ ]:
variants = {
  'terse':        'Refund order 4471.',
  'polite prose': 'Could you please go ahead and process a refund for the customer order '
                  'number 4471 at your earliest convenience? Thank you very much.',
  'JSON':         '{"action": "refund", "order_id": "4471", "reason": "customer_request"}',
  'markdown row': '| action | order_id | reason |\n| refund | 4471 | customer_request |',
  'enterprise':   'Please execute REFUND_TXN on SalesOrder__c record a0X5f000001ABCDEAA2 '
                  'per policy FIN-REF-07 (see KB article KB0043921).',
}

rows = [dict(form=k, tokens=ntok(v), chars=len(v), chars_per_token=round(len(v)/ntok(v),2))
        for k, v in variants.items()]
df = pd.DataFrame(rows).sort_values('tokens')
display(df)
print(f"\nSpread: {df.tokens.max()/df.tokens.min():.1f}x for the same instruction")
print('Note how the enterprise identifiers destroy the ~4 chars/token rule of thumb.')

### Try your own

Paste a real field name, SKU, or boilerplate disclaimer from your own system.
This is the moment in class where the room realises the lever is in their hands.

In [ ]:
mine = 'a0X5f000001ABCDEAA2'   # <-- replace with your own worst offender
toks = enc.encode(mine)
print(f'{mine!r} -> {len(toks)} tokens')
print('pieces:', [enc.decode([t]) for t in toks])

---
## 2. The 107× slide, computed live

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
import matplotlib.pyplot as plt

REQUESTS, T_IN, T_OUT = 1_000_000, 800, 200

rows = [dict(model=m, monthly=cost(m, T_IN*REQUESTS, T_OUT*REQUESTS)) for m in PRICES]
df = pd.DataFrame(rows).sort_values('monthly')
df['x_vs_cheapest'] = (df.monthly / df.monthly.min()).round(1)
display(df.style.format({'monthly': '${:,.0f}'}))

print(f'\nSPREAD: {df.x_vs_cheapest.max():.0f}x  '
      f'({usd(df.monthly.min())}/mo -> {usd(df.monthly.max())}/mo)')

ax = df.plot.barh(x='model', y='monthly', legend=False, figsize=(9,5),
                  color=['#0CA678']*3 + ['#1098AD']*2 + ['#F08C00']*3 + ['#E03131']*3)
ax.set_xlabel('USD / month'); ax.set_ylabel('')
ax.set_title(f'Identical assistant: {REQUESTS:,} req/mo, {T_IN} in / {T_OUT} out')
plt.tight_layout(); plt.show()

---
## 3. The forecast function — with the three multipliers

```
Monthly $ = Requests
          x [ fresh_in x P_in  +  cached_in x P_cache  +  out x P_out ] / 1e6
          x M_agent        <- steps / tool calls per user-visible task
          x M_context      <- long-context tier multiplier (1.0 flat, ~2.0 tiered)
          x (1 + R_retry)  <- retries, failed tool calls, regenerations
```

In [ ]:
def forecast(model, requests_per_month, tokens_in, tokens_out,
             agent_steps=1, cache_hit_rate=0.0, context_multiplier=1.0,
             retry_rate=0.0, verbose=True):
    """Monthly cost for a workload. agent_steps>1 assumes context is re-read
    and grows linearly with each step (the dominant agentic cost driver)."""
    # total input tokens across the whole trajectory: 1x + 2x + ... + n x base
    step_factor = agent_steps * (agent_steps + 1) / 2 if agent_steps > 1 else 1
    total_in  = tokens_in * step_factor
    total_out = tokens_out * agent_steps

    cached_in = total_in * cache_hit_rate
    fresh_in  = total_in - cached_in

    per_task = cost(model, inp=fresh_in, out=total_out, cache_r=cached_in)
    per_task *= context_multiplier * (1 + retry_rate)
    monthly = per_task * requests_per_month

    if verbose:
        print(f'  model            {model}')
        print(f'  input tokens     {total_in:,.0f}  ({fresh_in:,.0f} fresh / {cached_in:,.0f} cached)')
        print(f'  output tokens    {total_out:,.0f}')
        print(f'  cost per task    {usd(per_task)}')
        print(f'  MONTHLY          {usd(monthly)}')
    return monthly, per_task

### Scenario A — the chat assistant your CFO was shown

In [ ]:
forecast('claude-sonnet-5', 1_000_000, 800, 200);

### Scenario B — the same feature, six months later, now agentic

Nothing about the user experience changed. It just calls tools now.

In [ ]:
forecast('claude-sonnet-5', 1_000_000, 800, 200,
         agent_steps=5, retry_rate=0.08);

### Scenario C — same agent, with prompt caching switched on

One hour of engineering work.

In [ ]:
forecast('claude-sonnet-5', 1_000_000, 800, 200,
         agent_steps=5, retry_rate=0.08, cache_hit_rate=0.80);

### Scenario D — and now route the routine 70% to the volume tier

In [ ]:
mix = {'claude-haiku-4-5': 0.70, 'claude-sonnet-5': 0.20, 'claude-opus-5': 0.10}
blended = sum(
    share * forecast(m, 1_000_000, 800, 200, agent_steps=5,
                     retry_rate=0.08, cache_hit_rate=0.80, verbose=False)[0]
    for m, share in mix.items())
print(f'Blended monthly across a 70/20/10 tier mix: {usd(blended)}')

---
## 4. Best / base / worst — never hand finance a point estimate

Stanford Digital Economy Lab found identical agents on identical tasks varied up to **30×**
in cost between runs. A single number will be wrong.

In [ ]:
scenarios = {
  'best  (cached, routed, tight)': dict(model='claude-haiku-4-5', agent_steps=3,
                                        cache_hit_rate=0.85, retry_rate=0.03),
  'base  (expected)':              dict(model='claude-sonnet-5', agent_steps=5,
                                        cache_hit_rate=0.60, retry_rate=0.08),
  'worst (no cache, long ctx)':    dict(model='claude-sonnet-5', agent_steps=9,
                                        cache_hit_rate=0.0,  retry_rate=0.20,
                                        context_multiplier=2.0),
}
out = []
for name, kw in scenarios.items():
    m = kw.pop('model')
    monthly, per_task = forecast(m, 1_000_000, 800, 200, verbose=False, **kw)
    out.append(dict(scenario=name, model=m, cost_per_task=per_task, monthly=monthly))
df = pd.DataFrame(out)
display(df.style.format({'cost_per_task': '${:,.4f}', 'monthly': '${:,.0f}'}))
print(f'\nSpread best->worst: {df.monthly.max()/df.monthly.min():.1f}x')
print('This range, not the middle number, is what you take to a budget review.')

---
## 5. Optional — a live round trip to check the arithmetic

Confirms that the token counts you are forecasting with match what you are actually billed for.

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'   # adjust to a model id your key can call

r = client.messages.create(
    model=MODEL, max_tokens=150,
    messages=[{'role': 'user',
               'content': 'In two sentences: why are output tokens priced higher than input tokens?'}])

u = r.usage
print(r.content[0].text, '\n')
log_call('live round trip', MODEL, inp=u.input_tokens, out=u.output_tokens)
print('\nCompare u.input_tokens to your own tiktoken estimate — they will be close but not identical,')
print('because each provider tokenizes with its own encoding. Always trust the usage object.')

In [ ]:
ledger()

---
## Takeaways

| | |
|---|---|
| The 107× spread | Model tier is the largest single line on the bill |
| `agent_steps` | Moves the forecast more than any price negotiation will |
| `cache_hit_rate` | A large partial fix, never a cure — output is never cached |
| best/base/worst | The only honest way to forecast an agentic workload |